# Insurance Pricing Factor Analysis & Visualisation

This notebook extends the base pricing factor generation with:
- **Visual comparison** of factor distributions across model versions
- **Statistical analysis** (mean, spread, variance) of each model
- **Heatmap** of factor values across all models
- **Correlation** and coverage analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

## 1. Generate Synthetic Pricing Factors

In [ ]:
np.random.seed(42)

factor_counts = [2, 3, 4, 5, 6, 7, 8, 9]
factor_labels = [
    "Age", "Vehicle Type", "Location", "Driving Record",
    "Annual Mileage", "Credit Score", "Claim History", "Coverage Level", "Deductible"
]

pricing_inputs = {}
for i, count in enumerate(factor_counts):
    factors = np.random.uniform(0, 1, size=count)
    factors = np.round(np.sort(factors), 8)
    pricing_inputs[f"Model v{i}"] = factors

for name, vals in pricing_inputs.items():
    print(f"{name} ({len(vals)} factors): {' | '.join(f'{v:.4f}' for v in vals)}")

## 2. Statistical Summary Table

In [ ]:
stats = []
for name, vals in pricing_inputs.items():
    stats.append({
        "Model": name,
        "Num Factors": len(vals),
        "Min": vals.min(),
        "Max": vals.max(),
        "Range": vals.max() - vals.min(),
        "Mean": np.round(vals.mean(), 6),
        "Std Dev": np.round(vals.std(), 6),
        "Median": np.round(np.median(vals), 6)
    })

df_stats = pd.DataFrame(stats)
df_stats.set_index("Model", inplace=True)
df_stats

## 3. Bar Chart — Factor Values per Model

Each model version is plotted as a group of bars showing its individual factor values.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

max_factors = max(len(v) for v in pricing_inputs.values())
colors = cm.viridis(np.linspace(0.2, 0.9, max_factors))

x_positions = np.arange(len(pricing_inputs))
bar_width = 0.08

for factor_idx in range(max_factors):
    values = []
    positions = []
    for model_idx, (name, vals) in enumerate(pricing_inputs.items()):
        if factor_idx < len(vals):
            values.append(vals[factor_idx])
            positions.append(model_idx + factor_idx * bar_width - (len(vals) - 1) * bar_width / 2)
    label = factor_labels[factor_idx] if factor_idx < len(factor_labels) else f"Factor {factor_idx+1}"
    ax.bar(positions, values, width=bar_width, color=colors[factor_idx], label=label, edgecolor='white', linewidth=0.5)

ax.set_xticks(x_positions)
ax.set_xticklabels(list(pricing_inputs.keys()), fontsize=10)
ax.set_ylabel("Factor Value (normalised)", fontsize=11)
ax.set_title("Insurance Pricing Factor Values by Model Version", fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Heatmap — Factor Values Across Models

A heatmap where rows are model versions and columns are factor indices. Missing factors (shorter models) appear as grey.

In [ ]:
heatmap_data = np.full((len(pricing_inputs), max_factors), np.nan)
model_names = list(pricing_inputs.keys())

for i, (name, vals) in enumerate(pricing_inputs.items()):
    heatmap_data[i, :len(vals)] = vals

fig, ax = plt.subplots(figsize=(12, 5))
cax = ax.imshow(heatmap_data, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=10)
col_labels = [factor_labels[j] if j < len(factor_labels) else f"Factor {j+1}" for j in range(max_factors)]
ax.set_xticks(range(max_factors))
ax.set_xticklabels(col_labels, fontsize=9, rotation=45, ha='right')
ax.set_title("Heatmap of Pricing Factor Values", fontsize=13, fontweight='bold')

for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        if not np.isnan(heatmap_data[i, j]):
            ax.text(j, i, f"{heatmap_data[i,j]:.2f}", ha='center', va='center', fontsize=8)

fig.colorbar(cax, label="Factor Value")
plt.tight_layout()
plt.show()

## 5. Mean & Spread per Model Version

Shows how the average factor value and standard deviation change as the number of factors increases.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

means = df_stats["Mean"].values
stds = df_stats["Std Dev"].values
num_factors = df_stats["Num Factors"].values
model_labels = df_stats.index.tolist()

ax1.bar(model_labels, means, color='steelblue', edgecolor='white')
ax1.errorbar(model_labels, means, yerr=stds, fmt='none', ecolor='black', capsize=4)
ax1.set_ylabel("Mean Factor Value")
ax1.set_title("Mean Factor Value ± Std Dev", fontweight='bold')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', alpha=0.3)

ax2.plot(num_factors, df_stats["Range"].values, 'o-', color='darkorange', linewidth=2, markersize=8)
ax2.set_xlabel("Number of Factors")
ax2.set_ylabel("Range (Max - Min)")
ax2.set_title("Factor Range vs Model Complexity", fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Factor Coverage Distribution

Histogram showing the overall distribution of all generated factor values across every model.

In [ ]:
all_factors = np.concatenate(list(pricing_inputs.values()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_factors, bins=20, color='teal', edgecolor='white', alpha=0.85)
ax.axvline(all_factors.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {all_factors.mean():.3f}')
ax.set_xlabel("Factor Value")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of All Generated Factor Values", fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total factors generated: {len(all_factors)}")
print(f"Overall mean: {all_factors.mean():.4f}")
print(f"Overall std:  {all_factors.std():.4f}")